In [13]:
import pandas as pd

In [14]:
from Descriptive import Descriptive

In [15]:
obj=Descriptive()

In [16]:
from nsepy import get_history as gh
import datetime as dt
import yfinance as yf
stock_symbol = "RELIANCE.NS" #NSE stocks usually end with .NS
#dowload the stock data from NSE
stk_data=yf.download(stock_symbol, start="2024-05-01", end="2025-10-09")

[*********************100%***********************]  1 of 1 completed


In [17]:
stk_data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 358 entries, 2024-05-02 to 2025-10-08
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   (Close, RELIANCE.NS)   358 non-null    float64
 1   (High, RELIANCE.NS)    358 non-null    float64
 2   (Low, RELIANCE.NS)     358 non-null    float64
 3   (Open, RELIANCE.NS)    358 non-null    float64
 4   (Volume, RELIANCE.NS)  358 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 16.8 KB


In [18]:
stk_data=stk_data[["Open","High","Low","Close"]]

In [19]:
stk_data

Price,Open,High,Low,Close
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,
2024-05-02,1454.460329,1459.721831,1446.679164,1449.075317
2024-05-03,1453.472085,1457.374970,1399.275682,1416.912964
2024-05-06,1418.395221,1422.841601,1401.103743,1402.610596
2024-05-07,1399.102859,1403.820987,1375.413559,1384.775635
2024-05-08,1380.848112,1415.875659,1380.848112,1401.647339
...,...,...,...,...
2025-10-01,1360.708629,1372.255218,1356.428371,1362.400757
2025-10-03,1356.926092,1365.287457,1350.655159,1357.125244


In [21]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data)
print("Len:",data1.shape)

Len: (358, 4)


In [22]:
#DataFrame containing stock prices and a list of column names to use
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [29]:
#Training and test set split
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

286
X_train length: (286, 4)
X_test length: (72, 4)
y_train length: (286, 4)
y_test length: (72, 4)


In [30]:
from sklearn.metrics import mean_squared_error
import numpy as np

In [40]:

def cominbation(dataset,listt):
#Defines a function named cominbation
    print(listt)
    #Prints the selected columns
    datasetTwo=dataset[listt]
    #Selects only the columns listed in listt
    test_obs = 28
    #Specifies that the last 28 observations will be used as the test set
    train =datasetTwo[:-test_obs]
    #Creates the training dataset,it uses all rows except the last 28
    test = datasetTwo[-test_obs:]
    ##Creates the test dataset,it uses all rows except the last 28
    from statsmodels.tsa.api import VAR
    #Imports the VAR (Vector Autoregression) to create a model multiple variables together
    for i in [1,2,3,4,5,6,7,8,9,10]:
        #ests lag orders from 1 to 10
        model = VAR(train)
        #Creates a VAR model using the training data
        results = model.fit(i)
        #Fits the VAR model with lag i
        print('Order =', i)
        print('AIC: ', results.aic)
        print('BIC: ', results.bic)
        print()
        #Displays the lag order and its information criteria
        #AIC (Akaike Information Criterion): lower is better
        #BIC (Bayesian Information Criterion): lower is better
    x = model.select_order(maxlags=12)
    #Automatically tests lag orders from 1 to 12.
    #Computes AIC, BIC for each lag
    order=x.selected_orders["aic"]
    #elects the lag order that has the lowest AIC
    result = model.fit(order)
    #Fits the final VAR model using the selected lag
    #result.summary()
    lagged_Values = train.values[-order:]
    #Retrieves the last order rows from the training data
    #These values are used as the starting point for forecasting
    pred = result.forecast(y=lagged_Values,steps=28) 
    #Forecasts the next 28 observations
    #pred is a NumPy array with one forecast for each selected column
    preds=pd.DataFrame(pred,columns=listt)
    #Converts the forecast array into a pandas DataFrame and uses the original column names
    preds.to_csv("varforecasted_{}.csv".format(test_obs))
    #Saves the forecast to a CSV file
    from sklearn.metrics import mean_squared_error
    #Imports the function to calculate Mean Squared Error (MSE)
    mse = mean_squared_error(test, pred)
    #Computes the MSE between the actual test data and the predictions
    rmse = round(np.sqrt(mse))
    #Converts MSE into Root Mean Squared Error
    from sklearn.metrics import mean_absolute_percentage_error
    #Imports the function to calculate MAPE (Mean Absolute Percentage Error)
    mape=mean_absolute_percentage_error(test,pred)
    #Computes the average percentage error between actual and predicted values
    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Lag"].append(order)
    #Stores the selected variables
    performance["Test"].append(test_obs)
    #Stores the test size (28)
    perf=pd.DataFrame(performance)
    #Converts the performance dictionary into a DataFrame
    return perf,result,pred
    #Returns three objects perf,result,pred

In [41]:
listt=["Close","High","Open","Low"]

In [42]:
perf,result,pred=cominbation(data1,listt)

['Close', 'High', 'Open', 'Low']
Order = 1
AIC:  -31.2307133264469
BIC:  -30.99994993734262

Order = 2
AIC:  -31.206233791988666
BIC:  -30.789927420336745

Order = 3
AIC:  -31.176795254338657
BIC:  -30.574110456520135

Order = 4
AIC:  -31.129932568842484
BIC:  -30.34002759358808

Order = 5
AIC:  -31.078164821908487
BIC:  -30.100191544014034

Order = 6
AIC:  -31.016080750940922
BIC:  -29.84918460409144

Order = 7
AIC:  -30.9826467134433
BIC:  -29.62596662213114

Order = 8
AIC:  -30.972840240397492
BIC:  -29.42550855091966

Order = 9
AIC:  -30.921616671318038
BIC:  -29.18275908183753

Order = 10
AIC:  -30.858052467874696
BIC:  -28.926787957530387



In [43]:
data1

,Open,High,Low,Close
0,0.717233,0.696765,0.740022,0.691190
1,0.715098,0.691287,0.637262,0.616371
2,0.639310,0.610678,0.641225,0.583100
3,0.597626,0.566280,0.585534,0.541611
4,0.558184,0.594418,0.597315,0.580859
...,...,...,...,...
353,0.514670,0.492599,0.544379,0.489561
354,0.506497,0.476334,0.531864,0.477289
355,0.499615,0.489811,0.536395,0.504149
356,0.534026,0.532795,0.572862,0.526842


In [44]:
perf

,Model,RMSE,MaPe,Lag,Test
0,"[Close, High, Open, Low]",0.038,0.053783,1,28
1,"[Close, High, Open, Low]",0.000,0.053783,1,28
